# Análise por Contagem de Palavras — Relatório de Gestão MRE 2025
**Opção 1 — Léxica por janela.** Objetivo: em que sentido o MRE aborda `internet` no relatório de 2025?
PDF: `relatorios-gestao-mre/Relatório-de-gestão_2025 (1).pdf`
Léxicos em `config.py`: `ABORDAGEM_A` (Soberana/Multilateral) vs `ABORDAGEM_B` (Mercado/Inovação). Janela = ±50 tokens.


In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd()))
from config import TERMO_CENTRAL_REGEX, ABORDAGEM_A_TERMOS, ABORDAGEM_B_TERMOS, ABORDAGEM_A_NOME, ABORDAGEM_B_NOME, JANELA_TOKENS
from utils import extract_text, analisar_texto

ANO = 2025
PDF_FILE = "Relatório-de-gestão_2025 (1).pdf"
PDF_PATH = Path("/workspaces/governanca-digital_mre/relatorios-gestao-mre") / PDF_FILE
print(f"Ano: {ANO}")
print(f"PDF: {PDF_PATH} existe={PDF_PATH.exists()}")


Ano: 2025
PDF: /workspaces/governanca-digital_mre/relatorios-gestao-mre/Relatório-de-gestão_2025 (1).pdf existe=True


In [2]:
# 1. Extrair texto
texto = extract_text(PDF_PATH)
print(f"Caracteres extraídos: {len(texto):,}")
print(texto[:800].replace(chr(10), ' ') + "...")


Caracteres extraídos: 259,103
1        Relatório de Gestão Integrado2025 Ministério das Relações Exteriores  2    Brasil. Ministério das Relações Exteriores.   Relatório de gestão integrado 2025 [recurso eletrônico] /  Ministério das Relações Exteriores – Brasília : MRE, 2026.   121 p.   Brasil. Ministério das Relações Exteriores, relatório, 2025. 2.  Relatório de Gestão – Brasil 3. Desempenho institucional – MRE 4.  Governança e estratégia – Brasil. 5. COP 30 6. Cúpula de líderes dos  BRICS I. Título.   Modo de acesso: mre.gov.br   CDU 342.532(81)(047)   3  Relatório de Gestão Integrado  – 2025   Embaixadora Maria Laura da Rocha   Secretária-Geral das Relações Exteriores   Embaixadora Márcia Loureiro  Secretária de Comunidades Brasileiras e Assuntos  Consulares e Jurídicos  Embaixadora Susan Kleebank  Secretária de Ás...


In [3]:
# 2. Analisar contextos do termo central
resultado = analisar_texto(texto, TERMO_CENTRAL_REGEX, ABORDAGEM_A_TERMOS, ABORDAGEM_B_TERMOS, janela=JANELA_TOKENS)
print(f"Ocorrências de 'internet': {resultado['n_ocorrencias']}")
print(f"Tokens totais no documento: {resultado['n_tokens_total']:,}")
print(f"Total termos {ABORDAGEM_A_NOME} nas janelas: {resultado['total_A']}")
print(f"Total termos {ABORDAGEM_B_NOME} nas janelas: {resultado['total_B']}")
print(f"Contagem A detalhada: {resultado['cnt_A']}")
print(f"Contagem B detalhada: {resultado['cnt_B']}")


Ocorrências de 'internet': 0
Tokens totais no documento: 38,299
Total termos Soberana/Multilateral nas janelas: 0
Total termos Mercado/Inovação nas janelas: 0
Contagem A detalhada: {}
Contagem B detalhada: {}


In [4]:
# 3. Tabelas e Gráficos — requer pandas/matplotlib (já instalados)
import pandas as pd
import matplotlib.pyplot as plt

if resultado['n_ocorrencias'] == 0:
    print("Nenhuma ocorrência de 'internet' encontrada neste relatório.")
else:
    # --- Tabela resumo ---
    total = resultado['total_A'] + resultado['total_B']
    perc_A = resultado['total_A']/total*100 if total else 0
    perc_B = resultado['total_B']/total*100 if total else 0
    dominante = ABORDAGEM_A_NOME if resultado['total_A'] > resultado['total_B'] else ABORDAGEM_B_NOME if resultado['total_B'] > resultado['total_A'] else "Empate/Neutro"
    df_resumo = pd.DataFrame([{
        "Ano": ANO,
        "Ocorrências 'internet'": resultado['n_ocorrencias'],
        "Tokens documento": resultado['n_tokens_total'],
        f"Total {ABORDAGEM_A_NOME}": resultado['total_A'],
        f"Total {ABORDAGEM_B_NOME}": resultado['total_B'],
        "% A": f"{perc_A:.1f}%",
        "% B": f"{perc_B:.1f}%",
        "Enquadramento dominante": dominante,
    }])
    display(df_resumo.style.set_caption(f"Resumo MRE {ANO} — entorno de 'internet' (±{JANELA_TOKENS} tokens)").hide(axis='index'))

    # --- Gráfico 1: barras totais A vs B ---
    labels = [ABORDAGEM_A_NOME, ABORDAGEM_B_NOME]
    valores = [resultado['total_A'], resultado['total_B']]
    cores = ['#6a51a3', '#ff7f0e']
    plt.figure(figsize=(6,4))
    bars = plt.bar(labels, valores, color=cores, edgecolor='black', alpha=0.85)
    plt.title(f"MRE {ANO}: termos nas janelas de 'internet' (±{JANELA_TOKENS} tokens)", fontsize=12, fontweight='bold')
    plt.ylabel("Contagem")
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    for bar, v in zip(bars, valores):
        plt.text(bar.get_x()+bar.get_width()/2, v+0.15, str(v), ha='center', fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.show()

    # --- Gráficos 2 e 3: breakdown por termo ---
    df_a = pd.DataFrame(list(resultado['cnt_A'].items()), columns=['termo','qtd']).sort_values('qtd', ascending=True) if resultado['cnt_A'] else pd.DataFrame(columns=['termo','qtd'])
    df_b = pd.DataFrame(list(resultado['cnt_B'].items()), columns=['termo','qtd']).sort_values('qtd', ascending=True) if resultado['cnt_B'] else pd.DataFrame(columns=['termo','qtd'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=False)
    if not df_a.empty:
        axes[0].barh(df_a['termo'], df_a['qtd'], color='#9e9ac8', edgecolor='black')
        axes[0].set_title(f" breakdown {ABORDAGEM_A_NOME}", fontsize=11)
        axes[0].set_xlabel("qtd")
        for i, v in enumerate(df_a['qtd']):
            axes[0].text(v+0.05, i, str(v), va='center', fontsize=9)
    else:
        axes[0].text(0.5, 0.5, "Nenhum termo A", ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_title(f"{ABORDAGEM_A_NOME} — vazio")
    if not df_b.empty:
        axes[1].barh(df_b['termo'], df_b['qtd'], color='#fdae6b', edgecolor='black')
        axes[1].set_title(f" breakdown {ABORDAGEM_B_NOME}", fontsize=11)
        axes[1].set_xlabel("qtd")
        for i, v in enumerate(df_b['qtd']):
            axes[1].text(v+0.05, i, str(v), va='center', fontsize=9)
    else:
        axes[1].text(0.5, 0.5, "Nenhum termo B", ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title(f"{ABORDAGEM_B_NOME} — vazio")
    plt.suptitle(f"Detalhamento por termo — MRE {ANO}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\nTop termos A:")
    display(df_a.sort_values('qtd', ascending=False).style.hide(axis='index') if not df_a.empty else "(vazio)")
    print("Top termos B:")
    display(df_b.sort_values('qtd', ascending=False).style.hide(axis='index') if not df_b.empty else "(vazio)")


Nenhuma ocorrência de 'internet' encontrada neste relatório.


In [5]:
# 4. Detalhe por ocorrência — tabela interativa + gráfico empilhado
if resultado['n_ocorrencias'] == 0:
    print("Sem ocorrências para detalhar.")
    # ainda mostra tabela vazia para consistência
    display(pd.DataFrame([{"Ano": ANO, "Ocorrências": 0, "Obs": "Nenhuma menção a 'internet' neste ano"}]).style.hide(axis='index'))
else:
    rows = []
    for i, det in enumerate(resultado['detalhes'], 1):
        rows.append({
            "#": i,
            "trecho (300 chars)": det['janela_texto'][:300] + "...",
            f"A ({ABORDAGEM_A_NOME})": det['total_A'],
            f"B ({ABORDAGEM_B_NOME})": det['total_B'],
            "dominante": ABORDAGEM_A_NOME if det['total_A']>det['total_B'] else ABORDAGEM_B_NOME if det['total_B']>det['total_A'] else "empate",
            "A detalhe": str(det['contagem_A']) if det['contagem_A'] else "—",
            "B detalhe": str(det['contagem_B']) if det['contagem_B'] else "—",
        })
    df_det = pd.DataFrame(rows)
    display(df_det.style.set_caption(f"Janelas de 'internet' — MRE {ANO} ({len(rows)} ocorrências, janela ±{JANELA_TOKENS})").hide(axis='index'))

    plt.figure(figsize=(10, 3))
    x = range(1, len(resultado['detalhes'])+1)
    a_vals = [d['total_A'] for d in resultado['detalhes']]
    b_vals = [d['total_B'] for d in resultado['detalhes']]
    plt.bar(x, a_vals, label=ABORDAGEM_A_NOME, color='#6a51a3', edgecolor='black')
    plt.bar(x, b_vals, bottom=a_vals, label=ABORDAGEM_B_NOME, color='#ff7f0e', edgecolor='black')
    plt.xticks(x)
    plt.xlabel("Ocorrência #")
    plt.ylabel("termos na janela")
    plt.title(f"MRE {ANO}: empilhado A/B por ocorrência de 'internet'")
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

    for i, det in enumerate(resultado['detalhes'][:3], 1):
        print(f"\n--- Ocorrência {i} (A={det['total_A']} B={det['total_B']}) ---")
        print(det['janela_texto'][:500])


Sem ocorrências para detalhar.


Ano,Ocorrências,Obs
2025,0,Nenhuma menção a 'internet' neste ano


## Interpretação
- Se `total_A > total_B`, o entorno de `internet` está mais associado a enquadramento soberano/multilateral.
- Se `total_B > total_A`, mais associado a mercado/inovação.
- Comparar este notebook com `analise_2020/2022/...` para ver evolução temporal.
- Para aprofundar, ver `utils.extrair_contextos` e ajustar `JANELA_TOKENS` em `config.py`.
